# March Machine Learning Mania 2026 - EDA

**Competencia**: Predecir las probabilidades de victoria para todos los posibles enfrentamientos del torneo NCAA 2026 (Men's + Women's).  
**Metrica**: Log Loss  
**Deadline**: 19 de Marzo 2026  
**Premio**: $50,000  

---

## Basketball 101: Lo que necesitas saber

### Como funciona el NCAA Basketball

- **Temporada regular** (Nov-Mar): ~350 equipos Division I juegan ~30 partidos cada uno
- **Conferencias**: Los equipos pertenecen a conferencias (ACC, Big Ten, SEC, etc.). La fuerza de las conferencias varia enormemente
- **Partido**: Dos mitades de 20 min (hombres) o cuatro cuartos de 10 min (mujeres). No hay empates (tiempo extra hasta que haya ganador)
- **Ventaja de local**: Los equipos locales ganan ~60-65% de los partidos. Los partidos del torneo son en cancha neutral

### El Torneo NCAA ("March Madness")

- **68 equipos** son seleccionados: 32 campeones de conferencia + 36 invitaciones "at-large"
- **Seeds 1-16**: Un comite clasifica a los equipos. Seed 1 = mas fuerte, Seed 16 = mas debil
- **Formato**: Eliminacion directa. Primera ronda: 1 vs 16, 2 vs 15, ... 8 vs 9
- **Rondas**: First Round -> Round of 32 -> Sweet 16 -> Elite 8 -> Final Four -> Championship

### Por que esta competencia es diferente

1. **Tiny evaluation set**: Solo ~63 partidos hombres + ~63 mujeres se evaluan. Alta varianza
2. **Predicimos TODOS los matchups posibles**: ~2,278 por genero, pero solo ~63 se juegan
3. **Log Loss castiga predicciones extremas erroneas brutalmente**: Predecir 0.99 y perder = 4.61 de loss. Predecir 0.85 = 1.90 de loss
4. **Debemos hacer clip de predicciones**: Minimo [0.05, 0.95]

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

DATA_DIR = Path('../data')
print(f'Data directory: {DATA_DIR.resolve()}')

## 1. Inventario de datos

Cargamos TODOS los CSV y mostramos su forma y columnas.

In [ ]:
# Load all CSVs into a dictionary
data = {}
for f in sorted(DATA_DIR.glob('*.csv')):
    key = f.stem
    data[key] = pd.read_csv(f)
    print(f'{key:45s} | {str(data[key].shape):>15s} | {list(data[key].columns[:6])}')

print(f'\nTotal: {len(data)} archivos')

In [ ]:
# Quick aliases for key datasets
m_teams = data['MTeams']
w_teams = data['WTeams']
m_seasons = data['MSeasons']
w_seasons = data['WSeasons']
m_reg_compact = data['MRegularSeasonCompactResults']
m_reg_detailed = data['MRegularSeasonDetailedResults']
w_reg_compact = data['WRegularSeasonCompactResults']
w_reg_detailed = data['WRegularSeasonDetailedResults']
m_tourney_compact = data['MNCAATourneyCompactResults']
m_tourney_detailed = data['MNCAATourneyDetailedResults']
w_tourney_compact = data['WNCAATourneyCompactResults']
w_tourney_detailed = data['WNCAATourneyDetailedResults']
m_seeds = data['MNCAATourneySeeds']
w_seeds = data['WNCAATourneySeeds']
m_massey = data['MMasseyOrdinals']
conferences = data['Conferences']
m_team_conf = data['MTeamConferences']
w_team_conf = data['WTeamConferences']
sub_stage1 = data['SampleSubmissionStage1']
sub_stage2 = data['SampleSubmissionStage2']

print('Aliases creados OK')

## 2. Cobertura temporal

In [ ]:
# Season coverage per dataset
datasets_with_season = {
    'M Reg Compact': m_reg_compact,
    'M Reg Detailed': m_reg_detailed,
    'M Tourney Compact': m_tourney_compact,
    'M Tourney Detailed': m_tourney_detailed,
    'M Seeds': m_seeds,
    'M Massey': m_massey,
    'M Team Conf': m_team_conf,
    'W Reg Compact': w_reg_compact,
    'W Reg Detailed': w_reg_detailed,
    'W Tourney Compact': w_tourney_compact,
    'W Seeds': w_seeds,
    'W Team Conf': w_team_conf,
}

print(f'{"Dataset":25s} | {"Min Season":>10s} | {"Max Season":>10s} | {"# Seasons":>10s} | {"# Rows":>10s}')
print('-' * 80)
for name, df in datasets_with_season.items():
    print(f'{name:25s} | {df.Season.min():>10d} | {df.Season.max():>10d} | {df.Season.nunique():>10d} | {len(df):>10,d}')

In [ ]:
# Games per season (Men's regular season)
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

m_reg_compact.groupby('Season').size().plot(ax=axes[0], marker='o', markersize=3)
axes[0].set_title('Men\'s Regular Season Games per Season')
axes[0].set_ylabel('Number of Games')

w_reg_compact.groupby('Season').size().plot(ax=axes[1], marker='o', markersize=3, color='coral')
axes[1].set_title('Women\'s Regular Season Games per Season')
axes[1].set_ylabel('Number of Games')

plt.tight_layout()
plt.show()

# Note any COVID dips (2020/2021)
print(f"\nMen's 2020 games: {len(m_reg_compact[m_reg_compact.Season == 2020]):,}")
print(f"Men's 2021 games: {len(m_reg_compact[m_reg_compact.Season == 2021]):,}")
print(f"Men's 2025 games: {len(m_reg_compact[m_reg_compact.Season == 2025]):,}")
print(f"Men's 2026 games: {len(m_reg_compact[m_reg_compact.Season == 2026]):,}")

## 3. Equipos y conferencias

In [ ]:
print(f"Men's teams: {len(m_teams)}")
print(f"Women's teams: {len(w_teams)}")
print(f"Conferences: {len(conferences)}")

# Teams per conference (current season)
latest_season = m_team_conf.Season.max()
current_conf = m_team_conf[m_team_conf.Season == latest_season]
conf_sizes = current_conf.groupby('ConfAbbrev').size().sort_values(ascending=False)

print(f"\nConference sizes in {latest_season} (Men's, top 15):")
conf_with_names = conf_sizes.reset_index()
conf_with_names.columns = ['ConfAbbrev', 'NumTeams']
conf_with_names = conf_with_names.merge(conferences, on='ConfAbbrev', how='left')
print(conf_with_names.head(15).to_string(index=False))

In [ ]:
# Power conferences: which conferences produce the most tournament teams?
tourney_teams = m_seeds[['Season', 'TeamID']].merge(
    m_team_conf[['Season', 'TeamID', 'ConfAbbrev']], on=['Season', 'TeamID']
)

# Count tournament appearances per conference (last 10 years)
recent = tourney_teams[tourney_teams.Season >= 2015]
conf_tourney_count = recent.groupby('ConfAbbrev').size().sort_values(ascending=False).head(15)

fig, ax = plt.subplots(figsize=(12, 5))
conf_tourney_count.plot(kind='bar', ax=ax, color='steelblue')
ax.set_title('Tournament Appearances by Conference (2015-2025, Men\'s)')
ax.set_ylabel('Total Tournament Teams')
ax.set_xlabel('Conference')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# Define power conferences
POWER_CONFERENCES = ['acc', 'big_twelve', 'big_ten', 'sec', 'big_east', 'pac_twelve', 'aac']
print(f"\nPower conferences defined: {POWER_CONFERENCES}")

## 4. Analisis de scores y margenes

In [ ]:
# Score distributions
m_reg_compact['ScoreMargin'] = m_reg_compact['WScore'] - m_reg_compact['LScore']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Winning scores
axes[0].hist(m_reg_compact['WScore'], bins=50, alpha=0.7, label='Winner', color='steelblue')
axes[0].hist(m_reg_compact['LScore'], bins=50, alpha=0.7, label='Loser', color='coral')
axes[0].set_title('Score Distribution (Men\'s Regular Season)')
axes[0].legend()
axes[0].set_xlabel('Points')

# Score margin distribution
axes[1].hist(m_reg_compact['ScoreMargin'], bins=50, color='steelblue', alpha=0.7)
axes[1].set_title('Score Margin Distribution')
axes[1].set_xlabel('Margin (Winner - Loser)')
axes[1].axvline(m_reg_compact['ScoreMargin'].median(), color='red', linestyle='--', label=f"Median: {m_reg_compact['ScoreMargin'].median():.0f}")
axes[1].legend()

# Average score by season (trending down?)
avg_scores = m_reg_compact.groupby('Season').agg({'WScore': 'mean', 'LScore': 'mean'}).reset_index()
avg_scores['AvgTotal'] = avg_scores['WScore'] + avg_scores['LScore']
axes[2].plot(avg_scores['Season'], avg_scores['AvgTotal'], marker='o', markersize=3)
axes[2].set_title('Average Combined Score by Season')
axes[2].set_ylabel('Avg Total Points (W + L)')

plt.tight_layout()
plt.show()

print(f"Average margin: {m_reg_compact['ScoreMargin'].mean():.1f} points")
print(f"Median margin: {m_reg_compact['ScoreMargin'].median():.0f} points")
print(f"Max margin: {m_reg_compact['ScoreMargin'].max()} points")

In [ ]:
# Home vs Away vs Neutral win rates
loc_counts = m_reg_compact['WLoc'].value_counts()
total_games = len(m_reg_compact)

print("Win Location Distribution (Men's Regular Season):")
print(f"  Home (H): {loc_counts.get('H', 0):>6,d} ({loc_counts.get('H', 0)/total_games*100:.1f}%) - Winner was at home")
print(f"  Away (A): {loc_counts.get('A', 0):>6,d} ({loc_counts.get('A', 0)/total_games*100:.1f}%) - Winner was away (underdog won)")
print(f"  Neutral (N): {loc_counts.get('N', 0):>6,d} ({loc_counts.get('N', 0)/total_games*100:.1f}%) - Neutral court")

# Home advantage: since WLoc=H means the WINNER was home,
# home win rate = H / (H + A) (excluding neutral)
home_games = loc_counts.get('H', 0)
away_wins = loc_counts.get('A', 0)  # games where away team won
home_win_rate = home_games / (home_games + away_wins)
print(f"\nHome team win rate (excl. neutral): {home_win_rate:.1%}")
print(f"This is ~{home_win_rate*100 - 50:.1f}% advantage = crucial for Elo adjustments")

# Tournament games are all neutral
tourney_locs = m_tourney_compact['WLoc'].value_counts()
print(f"\nTournament game locations: {dict(tourney_locs)}")
print("→ Virtually all neutral, as expected")

## 5. Analisis de Seeds del torneo

**El seed es el feature mas predictivo de esta competencia.** Un comite de seleccion clasifica a los 68 equipos en seeds 1-16.

In [ ]:
# Parse seed strings
def parse_seed(seed_str):
    """Parse 'W01' -> (region='W', seed_num=1). Handle play-in: 'W16a' -> 16"""
    region = seed_str[0]
    seed_num = int(seed_str[1:3])
    return region, seed_num

m_seeds['Region'] = m_seeds['Seed'].apply(lambda x: parse_seed(x)[0])
m_seeds['SeedNum'] = m_seeds['Seed'].apply(lambda x: parse_seed(x)[1])

w_seeds['Region'] = w_seeds['Seed'].apply(lambda x: parse_seed(x)[0])
w_seeds['SeedNum'] = w_seeds['Seed'].apply(lambda x: parse_seed(x)[1])

print("Men's seed distribution (all years):")
print(m_seeds['SeedNum'].value_counts().sort_index())
print(f"\nSeeds per season: {m_seeds.groupby('Season').size().mode().values[0]}")

In [ ]:
# Historical seed matchup win rates (THE MOST IMPORTANT ANALYSIS)
# Merge seeds with tournament results
def compute_seed_win_rates(tourney_df, seeds_df):
    """Compute historical win rates for each seed matchup."""
    # Merge winner and loser seeds
    games = tourney_df.merge(
        seeds_df[['Season', 'TeamID', 'SeedNum']].rename(columns={'TeamID': 'WTeamID', 'SeedNum': 'WSeed'}),
        on=['Season', 'WTeamID']
    ).merge(
        seeds_df[['Season', 'TeamID', 'SeedNum']].rename(columns={'TeamID': 'LTeamID', 'SeedNum': 'LSeed'}),
        on=['Season', 'LTeamID']
    )
    
    # Create matchup key (higher seed num = weaker team)
    games['StrongSeed'] = games[['WSeed', 'LSeed']].min(axis=1)
    games['WeakSeed'] = games[['WSeed', 'LSeed']].max(axis=1)
    games['StrongWon'] = (games['WSeed'] == games['StrongSeed']).astype(int)
    
    # Win rate by seed matchup
    matchup_stats = games.groupby(['StrongSeed', 'WeakSeed']).agg(
        total_games=('StrongWon', 'count'),
        strong_wins=('StrongWon', 'sum')
    ).reset_index()
    matchup_stats['strong_win_rate'] = matchup_stats['strong_wins'] / matchup_stats['total_games']
    
    return games, matchup_stats

m_games_with_seeds, m_matchup_stats = compute_seed_win_rates(m_tourney_compact, m_seeds)
w_games_with_seeds, w_matchup_stats = compute_seed_win_rates(w_tourney_compact, w_seeds)

# Classic first-round matchups
first_round = [(1,16), (2,15), (3,14), (4,13), (5,12), (6,11), (7,10), (8,9)]

print("Historical First Round Win Rates (Higher Seed Wins):")
print(f"{'Matchup':>10s} | {'Men Win%':>10s} | {'Men Games':>10s} | {'Women Win%':>10s} | {'Women Games':>10s}")
print('-' * 60)

for strong, weak in first_round:
    m_row = m_matchup_stats[(m_matchup_stats.StrongSeed == strong) & (m_matchup_stats.WeakSeed == weak)]
    w_row = w_matchup_stats[(w_matchup_stats.StrongSeed == strong) & (w_matchup_stats.WeakSeed == weak)]
    
    m_wr = m_row['strong_win_rate'].values[0] if len(m_row) > 0 else float('nan')
    m_n = m_row['total_games'].values[0] if len(m_row) > 0 else 0
    w_wr = w_row['strong_win_rate'].values[0] if len(w_row) > 0 else float('nan')
    w_n = w_row['total_games'].values[0] if len(w_row) > 0 else 0
    
    print(f'{strong:>2d} vs {weak:>2d}   | {m_wr:>9.1%} | {m_n:>10d} | {w_wr:>9.1%} | {w_n:>10d}')

In [ ]:
# Seed difference vs win probability (KEY RELATIONSHIP)
def seed_diff_analysis(games_df):
    """Analyze how seed difference relates to win probability."""
    # For each game, compute seed diff (team with lower ID perspective)
    games_df = games_df.copy()
    games_df['TeamID1'] = games_df[['WTeamID', 'LTeamID']].min(axis=1)
    games_df['TeamID2'] = games_df[['WTeamID', 'LTeamID']].max(axis=1)
    
    # Get seeds for TeamID1 and TeamID2
    games_df['Seed1'] = np.where(games_df['WTeamID'] == games_df['TeamID1'], games_df['WSeed'], games_df['LSeed'])
    games_df['Seed2'] = np.where(games_df['WTeamID'] == games_df['TeamID2'], games_df['WSeed'], games_df['LSeed'])
    
    games_df['SeedDiff'] = games_df['Seed1'] - games_df['Seed2']  # Negative = Team1 is stronger
    games_df['Team1Won'] = (games_df['WTeamID'] == games_df['TeamID1']).astype(int)
    
    # Win rate by seed difference
    stats = games_df.groupby('SeedDiff').agg(
        win_rate=('Team1Won', 'mean'),
        count=('Team1Won', 'count')
    ).reset_index()
    
    return games_df, stats

m_games_diff, m_diff_stats = seed_diff_analysis(m_games_with_seeds)

fig, ax = plt.subplots(figsize=(14, 6))
scatter = ax.scatter(m_diff_stats['SeedDiff'], m_diff_stats['win_rate'], 
                     s=m_diff_stats['count'] * 2, alpha=0.6, c='steelblue')
ax.axhline(0.5, color='red', linestyle='--', alpha=0.5)
ax.axvline(0, color='red', linestyle='--', alpha=0.5)
ax.set_xlabel('Seed Difference (Team1 - Team2)')
ax.set_ylabel('Team1 Win Rate')
ax.set_title('Seed Difference vs Win Probability (Men\'s Tournament, bubble size = # games)')
plt.tight_layout()
plt.show()

print("\nKey insight: This S-shaped curve is essentially our target function!")
print("A logistic regression on seed difference alone is a very strong baseline.")

In [ ]:
# Men's vs Women's predictability comparison
def upset_rate_by_round(games_df, seeds_df, gender_label):
    """Compute upset rate (lower seed winning) by tournament round."""
    games = games_df.copy()
    games['Upset'] = games['WSeed'] > games['LSeed']  # Higher seed number won = upset
    
    # Round can be estimated from DayNum
    # Men's: DayNums roughly 134-135 (R64), 136-137 (R32), 138-139 (S16), 143 (E8), 145 (F4), 147-152 (Champ)
    upset_rate = games['Upset'].mean()
    return upset_rate

m_upset = upset_rate_by_round(m_games_with_seeds, m_seeds, "Men's")
w_upset = upset_rate_by_round(w_games_with_seeds, w_seeds, "Women's")

print(f"Overall upset rate (lower seed wins):")
print(f"  Men's:   {m_upset:.1%}")
print(f"  Women's: {w_upset:.1%}")
print(f"\n→ Women's tournament is {'more' if w_upset < m_upset else 'less'} predictable (fewer upsets)")

## 6. Massey Ordinals (Rankings)

La competencia incluye rankings de 100+ sistemas diferentes. Esto es una mina de oro de features.

In [ ]:
print(f"Massey Ordinals: {len(m_massey):,} rows")
print(f"Seasons: {m_massey.Season.min()} - {m_massey.Season.max()}")
print(f"Ranking systems: {m_massey.SystemName.nunique()}")

# List all systems
systems = m_massey.SystemName.unique()
print(f"\nAll {len(systems)} ranking systems:")
for i in range(0, len(systems), 8):
    print('  ' + ', '.join(systems[i:i+8]))

In [ ]:
# Which systems have the most coverage (appear in most seasons)?
system_coverage = m_massey.groupby('SystemName')['Season'].nunique().sort_values(ascending=False)

print(f"Top 20 ranking systems by season coverage:")
print(system_coverage.head(20).to_string())

# Systems available for 2026 (critical!)
systems_2026 = m_massey[m_massey.Season == 2026]['SystemName'].unique()
print(f"\nSystems available for 2026: {len(systems_2026)}")
print(f"Latest ranking day for 2026: {m_massey[m_massey.Season == 2026]['RankingDayNum'].max()}")

In [ ]:
# Which ranking systems best predict tournament outcomes?
# For each system, get end-of-regular-season ranking (highest DayNum < 134)
# Then check if higher-ranked team won tournament game

def evaluate_ranking_system(system_name, massey_df, tourney_df, seeds_df, max_day=133):
    """Evaluate how well a ranking system predicts tournament outcomes."""
    # Get latest pre-tournament ranking for each team
    sys_data = massey_df[massey_df.SystemName == system_name].copy()
    latest_day = sys_data.groupby(['Season', 'TeamID'])['RankingDayNum'].max().reset_index()
    latest_day = latest_day[latest_day.RankingDayNum <= max_day]
    
    rankings = latest_day.merge(sys_data, on=['Season', 'TeamID', 'RankingDayNum'])
    
    # Merge with tournament games
    games = tourney_df.merge(
        rankings[['Season', 'TeamID', 'OrdinalRank']].rename(columns={'TeamID': 'WTeamID', 'OrdinalRank': 'WRank'}),
        on=['Season', 'WTeamID'], how='inner'
    ).merge(
        rankings[['Season', 'TeamID', 'OrdinalRank']].rename(columns={'TeamID': 'LTeamID', 'OrdinalRank': 'LRank'}),
        on=['Season', 'LTeamID'], how='inner'
    )
    
    if len(games) == 0:
        return 0, 0
    
    # Higher ranked = lower OrdinalRank number
    correct = (games['WRank'] < games['LRank']).sum()
    accuracy = correct / len(games)
    return accuracy, len(games)

# Evaluate top systems (by coverage)
top_systems = system_coverage.head(30).index.tolist()
system_scores = []

for sys_name in top_systems:
    acc, n_games = evaluate_ranking_system(sys_name, m_massey, m_tourney_compact, m_seeds)
    system_scores.append({'System': sys_name, 'Accuracy': acc, 'Games': n_games})

sys_df = pd.DataFrame(system_scores).sort_values('Accuracy', ascending=False)
print("Top ranking systems by tournament prediction accuracy:")
print(sys_df.head(15).to_string(index=False))

## 7. Sample Submission Analysis

In [ ]:
# Parse submission format
print("Stage 1 (validation):")
print(f"  Shape: {sub_stage1.shape}")
print(f"  First 5 IDs: {sub_stage1['ID'].head().tolist()}")
print(f"  Last 5 IDs: {sub_stage1['ID'].tail().tolist()}")

# Parse seasons from IDs
s1_seasons = sub_stage1['ID'].apply(lambda x: int(x.split('_')[0])).unique()
print(f"  Seasons: {sorted(s1_seasons)}")

print(f"\nStage 2 (final prediction):")
print(f"  Shape: {sub_stage2.shape}")
print(f"  First 5 IDs: {sub_stage2['ID'].head().tolist()}")

s2_seasons = sub_stage2['ID'].apply(lambda x: int(x.split('_')[0])).unique()
print(f"  Seasons: {sorted(s2_seasons)}")

# How many matchups per season?
for s in sorted(s1_seasons):
    count = sub_stage1['ID'].apply(lambda x: int(x.split('_')[0]) == s).sum()
    print(f"  Season {s}: {count:,} matchups")

for s in sorted(s2_seasons):
    count = sub_stage2['ID'].apply(lambda x: int(x.split('_')[0]) == s).sum()
    print(f"  Season {s}: {count:,} matchups")

In [ ]:
# Verify TeamID1 < TeamID2 in submission
sample_ids = sub_stage2['ID'].head(10)
for id_str in sample_ids:
    parts = id_str.split('_')
    season, t1, t2 = int(parts[0]), int(parts[1]), int(parts[2])
    assert t1 < t2, f"TeamID1 ({t1}) >= TeamID2 ({t2})!"

print("Confirmed: TeamID1 < TeamID2 in all submission IDs")

# Check if submission includes both men and women
all_team_ids = set()
for id_str in sub_stage2['ID']:
    parts = id_str.split('_')
    all_team_ids.add(int(parts[1]))
    all_team_ids.add(int(parts[2]))

mens_ids = {t for t in all_team_ids if t < 3000}
womens_ids = {t for t in all_team_ids if t >= 3000}

print(f"\nUnique team IDs in Stage 2: {len(all_team_ids)}")
print(f"Men's teams (ID < 3000): {len(mens_ids)}")
print(f"Women's teams (ID >= 3000): {len(womens_ids)}")
print(f"\nMen's matchups: C({len(mens_ids)},2) = {len(mens_ids)*(len(mens_ids)-1)//2}")
print(f"Women's matchups: C({len(womens_ids)},2) = {len(womens_ids)*(len(womens_ids)-1)//2}")
print(f"Total expected: {len(mens_ids)*(len(mens_ids)-1)//2 + len(womens_ids)*(len(womens_ids)-1)//2}")
print(f"Actual: {len(sub_stage2)}")

## 8. Detailed Box Score Analysis

Los box scores detallados (disponibles desde 2003) contienen las estadisticas clave para calcular eficiencia.

In [ ]:
print("Detailed Results columns:")
print(m_reg_detailed.columns.tolist())
print(f"\nShape: {m_reg_detailed.shape}")
print(f"Seasons: {m_reg_detailed.Season.min()} - {m_reg_detailed.Season.max()}")

# Basic stats for winners vs losers
winner_stats = ['WFGM', 'WFGA', 'WFGM3', 'WFGA3', 'WFTM', 'WFTA', 'WOR', 'WDR', 'WAst', 'WTO', 'WStl', 'WBlk']
loser_stats = ['LFGM', 'LFGA', 'LFGM3', 'LFGA3', 'LFTM', 'LFTA', 'LOR', 'LDR', 'LAst', 'LTO', 'LStl', 'LBlk']
stat_names = ['FGM', 'FGA', 'FGM3', 'FGA3', 'FTM', 'FTA', 'OR', 'DR', 'Ast', 'TO', 'Stl', 'Blk']

print(f"\n{'Stat':>6s} | {'Winner Avg':>10s} | {'Loser Avg':>10s} | {'Diff':>8s}")
print('-' * 45)
for wc, lc, name in zip(winner_stats, loser_stats, stat_names):
    w_avg = m_reg_detailed[wc].mean()
    l_avg = m_reg_detailed[lc].mean()
    print(f'{name:>6s} | {w_avg:>10.1f} | {l_avg:>10.1f} | {w_avg - l_avg:>+8.1f}')

In [ ]:
# Calculate key efficiency metrics for a sample
df = m_reg_detailed.copy()

# Possessions estimate (for each team)
df['WPoss'] = df['WFGA'] - df['WOR'] + df['WTO'] + 0.475 * df['WFTA']
df['LPoss'] = df['LFGA'] - df['LOR'] + df['LTO'] + 0.475 * df['LFTA']
df['AvgPoss'] = (df['WPoss'] + df['LPoss']) / 2

# Offensive efficiency (points per 100 possessions)
df['WOffEff'] = df['WScore'] / df['WPoss'] * 100
df['LOffEff'] = df['LScore'] / df['LPoss'] * 100

# Effective FG%
df['WeFG'] = (df['WFGM'] + 0.5 * df['WFGM3']) / df['WFGA']
df['LeFG'] = (df['LFGM'] + 0.5 * df['LFGM3']) / df['LFGA']

# Turnover rate
df['WTORate'] = df['WTO'] / df['WPoss']
df['LTORate'] = df['LTO'] / df['LPoss']

# Offensive rebound rate
df['WORRate'] = df['WOR'] / (df['WOR'] + df['LDR'])
df['LORRate'] = df['LOR'] / (df['LOR'] + df['WDR'])

# FT rate
df['WFTRate'] = df['WFTA'] / df['WFGA']
df['LFTRate'] = df['LFTA'] / df['LFGA']

print("The Four Factors of Basketball (Winner vs Loser):")
print(f"{'Factor':>25s} | {'Winner':>8s} | {'Loser':>8s} | {'Diff':>8s} | Weight")
print('-' * 70)
factors = [
    ('Effective FG%', 'WeFG', 'LeFG', '~40%'),
    ('Turnover Rate', 'WTORate', 'LTORate', '~25%'),
    ('Off Rebound Rate', 'WORRate', 'LORRate', '~20%'),
    ('Free Throw Rate', 'WFTRate', 'LFTRate', '~15%'),
]
for name, w_col, l_col, weight in factors:
    w_val = df[w_col].mean()
    l_val = df[l_col].mean()
    print(f'{name:>25s} | {w_val:>8.3f} | {l_val:>8.3f} | {w_val - l_val:>+8.3f} | {weight}')

print(f"\nAverage possessions per game: {df['AvgPoss'].mean():.1f}")
print(f"Winner off efficiency: {df['WOffEff'].mean():.1f} pts/100 poss")
print(f"Loser off efficiency: {df['LOffEff'].mean():.1f} pts/100 poss")

In [ ]:
# Visualize the Four Factors
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# eFG%
axes[0, 0].hist(df['WeFG'], bins=50, alpha=0.6, label='Winner', color='steelblue')
axes[0, 0].hist(df['LeFG'], bins=50, alpha=0.6, label='Loser', color='coral')
axes[0, 0].set_title('Effective FG% (~40% of winning)')
axes[0, 0].legend()

# TO Rate
axes[0, 1].hist(df['WTORate'], bins=50, alpha=0.6, label='Winner', color='steelblue')
axes[0, 1].hist(df['LTORate'], bins=50, alpha=0.6, label='Loser', color='coral')
axes[0, 1].set_title('Turnover Rate (~25% of winning)')
axes[0, 1].legend()

# OR Rate
axes[1, 0].hist(df['WORRate'], bins=50, alpha=0.6, label='Winner', color='steelblue')
axes[1, 0].hist(df['LORRate'], bins=50, alpha=0.6, label='Loser', color='coral')
axes[1, 0].set_title('Off Rebound Rate (~20% of winning)')
axes[1, 0].legend()

# FT Rate
axes[1, 1].hist(df['WFTRate'], bins=50, alpha=0.6, label='Winner', color='steelblue')
axes[1, 1].hist(df['LFTRate'], bins=50, alpha=0.6, label='Loser', color='coral')
axes[1, 1].set_title('Free Throw Rate (~15% of winning)')
axes[1, 1].legend()

plt.suptitle('The Four Factors of Basketball', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 9. Coaches Analysis

In [ ]:
coaches = data['MTeamCoaches']
print(f"Coaches dataset: {coaches.shape}")
print(coaches.head())

# How many unique coaches?
print(f"\nUnique coaches: {coaches.CoachName.nunique()}")

# Coaches with most seasons
coach_seasons = coaches.groupby('CoachName')['Season'].nunique().sort_values(ascending=False)
print(f"\nCoaches with most seasons:")
print(coach_seasons.head(10))

## 10. Resumen y proximos pasos

### Hallazgos clave:

1. **35 archivos CSV** con datos de Men's (M prefix) y Women's (W prefix)
2. **Seeds son el predictor #1**: La relacion seed_diff → win_probability es un sigmoid claro
3. **Massey Ordinals**: 100+ sistemas de ranking, algunos mucho mejores que otros
4. **Four Factors**: eFG%, TO Rate, OR Rate, FT Rate explican ~90% de ganar
5. **Home advantage**: ~60-65% win rate en casa, pero el torneo es neutral
6. **Two stages**: Stage 1 (2022-2025 validation), Stage 2 (2026 prediction)
7. **Women's is more predictable**: Menos upsets que en men's
8. **COVID gap**: 2020 tiene datos parciales

### Feature engineering priorities:

1. **Elo ratings** (game-by-game, margin-adjusted, home-adjusted)
2. **KenPom-style efficiency** (off/def efficiency per 100 possessions + Four Factors)
3. **Seed features** (seed_diff, historical win rates by matchup)
4. **Massey ordinals** (top 10 systems aggregated: mean, median, std)
5. **Conference strength** (mean Elo of conference, power conference flag)
6. **Win/loss record**, strength of schedule, recent form